# 玄女底座遥感数据集 — 快速开始指南> **XuanNv Harbin Dataset Quick Start**本 Notebook 演示如何：1. 从 ModelScope 下载 **mini_test**（patch_000186）最小化数据集2. 加载预计算嵌入向量（128-dim embedding）3. 使用下游任务模型进行推理：   - **变化检测**（ChangeDetectionHeadV3）   - **土地覆盖分类**（WorldCover / Dynamic World）   - **水体提取**（JRC Water）   - **建筑物提取**（Building Extraction）4. 可视化推理结果## 相关资源- **数据集**: [WeijieWu/xuannv_embdding](https://modelscope.cn/datasets/WeijieWu/xuannv_embdding)- **下游模型**: [WeijieWu/xuannv-downstream-models](https://modelscope.cn/models/WeijieWu/xuannv-downstream-models)- **展示平台**: http://60.31.21.42:22060/

In [ ]:
# 安装必要的依赖# !pip install torch numpy scipy scikit-learn joblib matplotlib pillow modelscope -qimport sysprint(f"Python: {sys.version}")

## 第一步：下载数据与模型我们提供两种下载方式：- **mini_test**: 只包含 patch_000186 的数据（~39MB），适合快速体验- **完整数据集**: 包含全部 424 个 patch 的数据（~16GB），适合正式训练本节演示下载 mini_test 和下游任务模型。

In [ ]:
import osfrom pathlib import Path# 设置工作目录WORK_DIR = Path("./xuannv_demo")WORK_DIR.mkdir(exist_ok=True)DATA_DIR = WORK_DIR / "mini_test"MODEL_DIR = WORK_DIR / "models"DATA_DIR.mkdir(exist_ok=True)MODEL_DIR.mkdir(exist_ok=True)print(f"工作目录: {WORK_DIR.absolute()}")print(f"数据目录: {DATA_DIR.absolute()}")print(f"模型目录: {MODEL_DIR.absolute()}")

In [ ]:
# 方式一：使用 ModelScope Python SDK 下载from modelscope.hub.snapshot_download import dataset_snapshot_download, model_snapshot_download# 下载 mini_test 数据（只下载 mini_test 子目录）print("正在下载 mini_test 数据...")dataset_path = dataset_snapshot_download(    dataset_id="WeijieWu/xuannv_embdding",    local_dir=str(DATA_DIR),    allow_file_pattern=["mini_test/**"],  # 只下载 mini_test)print(f"数据下载完成: {dataset_path}")# 下载下游任务模型print("正在下载模型...")model_path = model_snapshot_download(    model_id="WeijieWu/xuannv-downstream-models",    local_dir=str(MODEL_DIR),)print(f"模型下载完成: {model_path}")

In [ ]:
# 方式二：使用 ModelScope CLI 下载（在终端中执行）# modelscope download --dataset WeijieWu/xuannv_embdding --local_dir ./xuannv_demo/mini_test --include mini_test/**# modelscope download --model WeijieWu/xuannv-downstream-models --local_dir ./xuannv_demo/models

## 第二步：理解数据格式### 嵌入向量（Embedding）嵌入向量是遥感影像经过 AlphaEarth Foundations 编码器后的 128 维特征表示：| 属性 | 值 ||------|-----|| 文件格式 | NumPy `.npy` || 数组形状 | `(128, 64, 64)` = `(embedding_dim, H, W)` || 数据类型 | `float32` || 空间分辨率 | 64×64（原始 128×128 经编码器下采样） || 归一化 | 已做 L2 归一化 + vMF 采样，位于单位超球面 || 适用距离 | 余弦距离（推荐） |命名规则：`patch_{6位编号}_{YYYY-MM}.npy`### 原始数据（Raw Data）mini_test 包含 patch_000186 的多源遥感数据（含高分辨率数据）：| 传感器 | 类型 | 说明 ||--------|------|------|| **S2** | 光学 | Sentinel-2，10m分辨率，6波段 || **S1** | SAR | Sentinel-1，C波段，VV/VH || **S2_HR** | 光学 | 高分光学，更高空间分辨率 || **S1_HR** | SAR | 高分雷达，更高空间分辨率 || **Landsat** | 光学 | 30m分辨率 || **DEM** | 高程 | 数字高程模型 || **WorldCover** | 标签 | ESA 11类土地覆盖 || **Dynamic World** | 标签 | Google 9类土地利用 || **JRC Water** | 标签 | 水体掩码 |

In [ ]:
import numpy as npfrom glob import glob# 嵌入向量路径EMB_V5_DIR = DATA_DIR / "mini_test" / "embeddings" / "v5_mixed_scale" / "monthly_embeddings_2025"EMB_V4_DIR = DATA_DIR / "mini_test" / "embeddings" / "v4_official" / "monthly_embeddings_2025"# 列出 patch_000186 的所有月份v5_files = sorted(glob(str(EMB_V5_DIR / "patch_000186_*.npy")))print(f"V5 版本可用月份: {[f.split('_')[-1].replace('.npy', '') for f in v5_files]}")# 加载一个嵌入向量emb = np.load(v5_files[0])print(f"\n加载文件: {Path(v5_files[0]).name}")print(f"形状: {emb.shape}")print(f"数据类型: {emb.dtype}")print(f"值范围: [{emb.min():.3f}, {emb.max():.3f}]")print(f"L2范数均值: {np.linalg.norm(emb, axis=0).mean():.3f}")

In [ ]:
import matplotlib.pyplot as plt# 使用 PCA 将 128 维降至 3 维进行 RGB 可视化from sklearn.decomposition import PCAemb_3d = emb.transpose(1, 2, 0).reshape(-1, 128)pca = PCA(n_components=3)emb_pca = pca.fit_transform(emb_3d)emb_pca = emb_pca.reshape(64, 64, 3)# 归一化到 [0, 1] 用于显示emb_pca = (emb_pca - emb_pca.min()) / (emb_pca.max() - emb_pca.min())fig, axes = plt.subplots(1, 3, figsize=(15, 4))axes[0].imshow(emb_pca)axes[0].set_title("Embedding PCA-RGB 可视化")axes[0].axis('off')# 显示第一个通道axes[1].imshow(emb[0], cmap='viridis')axes[1].set_title("Channel 0 (第一维特征)")axes[1].axis('off')# 显示嵌入向量的统计分布axes[2].hist(emb.flatten(), bins=100, alpha=0.7)axes[2].set_title("嵌入向量数值分布")axes[2].set_xlabel("Value")axes[2].set_ylabel("Frequency")plt.tight_layout()plt.show()print(f"PCA 解释方差比: {pca.explained_variance_ratio_}")print(f"累计解释方差: {pca.explained_variance_ratio_.sum():.3f}")

## 第三步：变化检测（Change Detection）变化检测头 **ChangeDetectionHeadV3** 基于两个时间点的嵌入向量，计算像素级变化概率。### 模型架构- **输入**: 两期嵌入向量 `emb_before`, `emb_after`，形状 `[B, 128, 64, 64]`- **特征融合**: 拼接四种特征图  1. `|emb_before - emb_after|`（差异绝对值）  2. `emb_before * emb_after`（逐元素乘积）  3. `emb_before`（前期嵌入）  4. `emb_after`（后期嵌入）  → 总通道数: `4 × 128 = 512`- **ECA 注意力**: Efficient Channel Attention，轻量级通道注意力增强变化区域响应- **输出**: Sigmoid 概率图 `[B, 1, 64, 64]`，值域 `[0, 1]`### 原理说明嵌入向量位于单位超球面上（已 L2 归一化）。对于未变化区域，两期嵌入向量方向相近，cosine similarity 接近 1；对于变化区域，方向差异大，cosine similarity 降低。变化检测头学习将这些差异映射为变化概率。

In [ ]:
import mathimport torchimport torch.nn as nnimport torch.nn.functional as F# ---------- ECA 通道注意力模块 ----------class ECA(nn.Module):    """Efficient Channel Attention (ECA) module.        轻量级通道注意力，参数量极少 (~100 params)，在遥感变化检测中    能有效抑制背景、增强变化区域响应。    """    def __init__(self, channels: int, gamma: float = 2.0, b: float = 1.0) -> None:        super().__init__()        kernel_size = int(abs((math.log(channels, 2) + b) / gamma))        kernel_size = kernel_size if kernel_size % 2 else kernel_size + 1        self.avg_pool = nn.AdaptiveAvgPool2d(1)        self.conv = nn.Conv1d(1, 1, kernel_size, padding=kernel_size // 2, bias=False)        self.sigmoid = nn.Sigmoid()    def forward(self, x: torch.Tensor) -> torch.Tensor:        y = self.avg_pool(x)        y = self.conv(y.squeeze(-1).transpose(-1, -2))        y = y.transpose(-1, -2).unsqueeze(-1)        return x * self.sigmoid(y)# ---------- ChangeDetectionHeadV3 ----------class ChangeDetectionHeadV3(nn.Module):    """V2 + ECA 通道注意力增强版.        在 residual block 之间插入 ECA，提升特征判别性。    """    def __init__(self, embedding_dim: int = 128, hidden_dim: int = 64, dropout: float = 0.3) -> None:        super().__init__()        in_dim = embedding_dim * 4        self.encoder = nn.Sequential(            nn.Conv2d(in_dim, hidden_dim, 3, padding=1),            nn.BatchNorm2d(hidden_dim),            nn.ReLU(),        )        self.res1 = nn.Sequential(            nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1),            nn.BatchNorm2d(hidden_dim),            nn.ReLU(),            nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1),            nn.BatchNorm2d(hidden_dim),        )        self.eca = ECA(hidden_dim)        self.res2 = nn.Sequential(            nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1),            nn.BatchNorm2d(hidden_dim),            nn.ReLU(),            nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1),            nn.BatchNorm2d(hidden_dim),        )        self.out = nn.Sequential(            nn.ReLU(),            nn.Conv2d(hidden_dim, hidden_dim // 2, 3, padding=1),            nn.BatchNorm2d(hidden_dim // 2),            nn.ReLU(),            nn.Dropout2d(dropout),            nn.Conv2d(hidden_dim // 2, 1, 1),        )    def forward(self, emb_before: torch.Tensor, emb_after: torch.Tensor) -> torch.Tensor:        diff = emb_before - emb_after        feat = torch.cat([            torch.abs(diff),            emb_before * emb_after,            emb_before,            emb_after,        ], dim=1)        x = self.encoder(feat)        x = F.relu(self.res1(x) + x)        x = self.eca(x)        x = F.relu(self.res2(x) + x)        return self.out(x)print("ChangeDetectionHeadV3 模型定义加载完成")print(f"模型参数量: {sum(p.numel() for p in ChangeDetectionHeadV3().parameters()):,}")

In [ ]:
# 加载变化检测模型权重CKPT_PATH = MODEL_DIR / "cd_head" / "monthly_cd_head_v5_final.pt"checkpoint = torch.load(str(CKPT_PATH), map_location="cpu", weights_only=False)cfg = checkpoint["config"]print(f"Checkpoint config: {cfg}")head = ChangeDetectionHeadV3(    embedding_dim=cfg["embedding_dim"],    hidden_dim=cfg["hidden_dim"],    dropout=cfg.get("dropout", 0.4),)head.load_state_dict(checkpoint["cd_head"])head.eval()print(f"\n模型加载成功！")print(f"训练指标: {checkpoint.get('metrics', 'N/A')}")

In [ ]:
# 加载两期嵌入向量并运行变化检测emb_before = np.load(str(EMB_V5_DIR / "patch_000186_2025-04.npy"))emb_after = np.load(str(EMB_V5_DIR / "patch_000186_2025-10.npy"))with torch.no_grad():    eb = torch.from_numpy(emb_before).unsqueeze(0).float()    ea = torch.from_numpy(emb_after).unsqueeze(0).float()    logits = head(eb, ea)    change_prob = torch.sigmoid(logits).squeeze().numpy()print(f"变化概率图形状: {change_prob.shape}")print(f"概率范围: [{change_prob.min():.4f}, {change_prob.max():.4f}]")print(f"平均变化概率: {change_prob.mean():.4f}")print(f"变化像素比例 (>0.5): {(change_prob > 0.5).mean():.2%}")

In [ ]:
# 可视化变化检测结果fig, axes = plt.subplots(1, 3, figsize=(15, 4))# 前期嵌入 PCA-RGBemb_b_pca = PCA(n_components=3).fit_transform(emb_before.transpose(1,2,0).reshape(-1,128)).reshape(64,64,3)emb_b_pca = (emb_b_pca - emb_b_pca.min()) / (emb_b_pca.max() - emb_b_pca.min())axes[0].imshow(emb_b_pca)axes[0].set_title("2025-04 嵌入向量 PCA-RGB")axes[0].axis('off')# 后期嵌入 PCA-RGBemb_a_pca = PCA(n_components=3).fit_transform(emb_after.transpose(1,2,0).reshape(-1,128)).reshape(64,64,3)emb_a_pca = (emb_a_pca - emb_a_pca.min()) / (emb_a_pca.max() - emb_a_pca.min())axes[1].imshow(emb_a_pca)axes[1].set_title("2025-10 嵌入向量 PCA-RGB")axes[1].axis('off')# 变化概率图im = axes[2].imshow(change_prob, cmap='coolwarm', vmin=0, vmax=1)axes[2].set_title("变化检测概率图 (2025-04 → 2025-10)")axes[2].axis('off')plt.colorbar(im, ax=axes[2], label='Change Probability')plt.tight_layout()plt.show()

## 第四步：分类任务推理本项目提供 4 个下游分类任务，基于 128-dim 嵌入向量做像素级分类：| 任务 | 类别数 | 模型 | 指标 (balanced_accuracy / f1) ||------|--------|------|------------------------------|| **WorldCover** | 11 | Linear Probe | 0.525 / 0.474 (macro) || **Dynamic World** | 9 | MLP | 0.502 / 0.496 (macro) || **JRC Water** | 2 | MLP | 0.822 / 0.862 (binary) || **Building** | 2 | MLP | 0.957 / 0.956 (binary) |### 推理流程1. 加载嵌入向量 `[128, 64, 64]`2. Reshape 为 `[4096, 128]`（展平空间维度）3. `scaler.transform()` 标准化4. `model.predict()` 得到分类标签5. Reshape 回 `[64, 64]`6. 后处理：二值任务做开运算+小区域过滤，多分类做3×3多数投票

In [ ]:
import joblibfrom scipy import ndimage# 模型文件映射MODEL_FILES = {    "worldcover": "sklearn_models/worldcover_linear_probe.pkl",    "dynamic_world": "sklearn_models/dynamic_world_sklearn_mlp.pkl",    "jrc_water": "sklearn_models/jrc_water_sklearn_mlp.pkl",    "building": "sklearn_models/building_sklearn_mlp.pkl",}def load_classifier(task: str):    """加载分类模型及其元数据."""    path = MODEL_DIR / MODEL_FILES[task]    data = joblib.load(str(path))    return datadef postprocess(pred: np.ndarray, n_classes: int) -> np.ndarray:    """后处理：去噪和平滑."""    pred = pred.astype(np.int32)    if n_classes == 2:        # 二值任务：开运算 + 小区域过滤        fg = (pred == 1).astype(np.uint8)        fg = ndimage.binary_opening(fg, structure=np.ones((3, 3))).astype(np.uint8)        labeled, num = ndimage.label(fg)        if num > 0:            sizes = ndimage.sum(fg, labeled, range(1, num + 1))            remove = np.where(sizes < 5)[0] + 1            if len(remove) > 0:                fg[np.isin(labeled, remove)] = 0        return np.where(fg, 1, 0).astype(np.int32)    else:        # 多分类：3×3 多数投票        def _majority(v):            vals, counts = np.unique(v, return_counts=True)            return int(vals[np.argmax(counts)])        return ndimage.generic_filter(pred, _majority, size=3, mode="nearest")def classify(task: str, emb: np.ndarray, apply_postprocess: bool = True):    """运行分类推理."""    data = load_classifier(task)    scaler = data["scaler"]    model = data["model"]    colors = data["colors"]    class_names = data["class_names"]        D, H, W = emb.shape    flat = emb.reshape(D, -1).T    flat_s = scaler.transform(flat)    pred = model.predict(flat_s).reshape(H, W).astype(np.int32)        if apply_postprocess:        n_classes = len(data["classes"])        pred = postprocess(pred, n_classes)        return pred, colors, class_namesprint("分类推理函数定义完成")

In [ ]:
# 加载一个嵌入向量用于分类emb_apr = np.load(str(EMB_V5_DIR / "patch_000186_2025-04.npy"))# 运行 4 个分类任务results = {}for task in ["worldcover", "dynamic_world", "jrc_water", "building"]:    pred, colors, class_names = classify(task, emb_apr)    results[task] = {"pred": pred, "colors": colors, "class_names": class_names}    print(f"{task:15s}: 类别数={len(class_names)}, 唯一值={np.unique(pred)}")

In [ ]:
# 可视化 4 个分类任务结果fig, axes = plt.subplots(2, 3, figsize=(16, 10))axes = axes.flatten()# 显示嵌入向量 PCA-RGB 作为参考emb_pca = PCA(n_components=3).fit_transform(emb_apr.transpose(1,2,0).reshape(-1,128)).reshape(64,64,3)emb_pca = (emb_pca - emb_pca.min()) / (emb_pca.max() - emb_pca.min())axes[0].imshow(emb_pca)axes[0].set_title("Embedding PCA-RGB (参考)")axes[0].axis('off')tasks = ["worldcover", "dynamic_world", "jrc_water", "building"]titles = ["WorldCover (11类)", "Dynamic World (9类)", "JRC Water (水体)", "Building (建筑物)"]for i, (task, title) in enumerate(zip(tasks, titles), 1):    pred = results[task]["pred"]    colors = np.array(results[task]["colors"], dtype=np.uint8)        # 将类别索引映射为 RGB    rgb = colors[pred]        axes[i].imshow(rgb)    axes[i].set_title(title)    axes[i].axis('off')# 隐藏多余的子图axes[5].axis('off')plt.tight_layout()plt.show()# 打印类别标签说明for task in tasks:    names = results[task]["class_names"]    print(f"\n{task}: {', '.join(names)}")

## 第五步：下载完整数据集（可选）mini_test 只包含 patch_000186 的数据，适合快速体验。如需全部 424 个 patch 的数据，请下载完整数据集。### 数据集结构```xuannv_embdding/├── README.md├── raw_data/│   ├── s2.tar.gz              # Sentinel-2 光学影像 (~4GB)│   ├── s1.tar.gz              # Sentinel-1 SAR (~4GB)│   ├── landsat.tar.gz         # Landsat 光学影像 (~449MB)│   ├── dynamic_world.tar.gz   # Dynamic World 标签 (~24MB)│   ├── dem/                   # 数字高程模型│   ├── jrc_water/             # JRC 水体数据│   ├── worldcover/            # ESA WorldCover│   └── scene_index.json       # 时间索引├── embeddings/│   ├── v4_official.tar.gz     # V4 嵌入向量 (~4.2GB)│   └── v5_mixed_scale.tar.gz  # V5 混合尺度嵌入向量 (~4.2GB)├── annotations/               # 变化检测人工标注 (shapefile)└── mini_test/                 # patch_000186 最小化测试集```### 下载命令```bash# 使用 ModelScope CLI 下载全部数据modelscope download --dataset WeijieWu/xuannv_embdding --local_dir ./xuannv_full# 解压 tar.gz 文件cd xuannv_full/raw_datatar xzf s2.tar.gztar xzf s1.tar.gztar xzf landsat.tar.gztar xzf dynamic_world.tar.gzcd ../embeddingstar xzf v4_official.tar.gztar xzf v5_mixed_scale.tar.gz```### 使用完整数据运行推理只需将 `EMB_V5_DIR` 指向完整数据的路径即可：```pythonEMB_V5_DIR = Path("./xuannv_full/embeddings/v5_mixed_scale/monthly_embeddings_2025")RAW_DIR = Path("./xuannv_full/raw_data")```

## 总结本 Notebook 演示了：1. ✅ 从 ModelScope 下载 **mini_test** 数据和下游任务模型2. ✅ 理解嵌入向量格式 `[128, 64, 64]`3. ✅ 使用 **ChangeDetectionHeadV3** 进行变化检测推理4. ✅ 使用 **sklearn 分类模型** 进行土地覆盖/土地利用/水体/建筑物提取5. ✅ 可视化推理结果### 关键要点- 嵌入向量已 L2 归一化，推荐使用**余弦距离**衡量变化强度- 变化检测需要**两期**嵌入向量，分类任务只需**一期**- 所有推理基于 CPU 即可运行（sklearn 模型纯 CPU，PyTorch 模型也可 CPU 推理）- mini_test 的 patch_000186 数据可以直接用于验证模型效果### 下一步建议- 使用完整数据集训练自己的下游任务头- 尝试不同月份组合的变化检测- 基于 annotations/ 中的 shapefile 标注评估模型性能